# 01 — CMS Medicare MUP 2023: Exploration & Contract Derivation

Profiling the **Medicare Physician & Other Practitioners — by Provider** extract
(2023 service year) to decide *what the data contract should assert*.

Every expectation in `pipelines/build_suites.py` traces back to something
observed in this notebook. The mapping is summarised at the bottom.

**Prerequisite:** `python pipelines/download_data.py` — the CSV lives in
`data/raw/` and is gitignored, so outputs are not committed with this notebook.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

DATA_FILE = Path("../data/raw/mup_phy_r25_p05_v20_d23_prov.csv")
SAMPLE_ROWS = 200_000  # full file is ~1.3M rows; a sample is enough to profile

assert DATA_FILE.exists(), f"Missing {DATA_FILE} — run pipelines/download_data.py first"
print(f"{DATA_FILE.name}: {DATA_FILE.stat().st_size / 1e6:,.0f} MB")

## 1. Load

Two columns need explicit dtypes, and both reasons are load-bearing for the
Postgres leg of the pipeline:

- `Rndrng_NPI` — read as string so the 10-digit identity is preserved rather
  than becoming an integer.
- `Rndrng_Prvdr_Zip5` — Canadian providers (`state = ZZ`) have alphanumeric
  postal codes like `K1H8`, which Pandas would otherwise infer as int64 and
  Postgres would reject with *invalid input syntax for type bigint*.

In [ ]:
DTYPES = {
    "Rndrng_NPI": str,
    "Rndrng_Prvdr_Zip5": str,
}

df = pd.read_csv(DATA_FILE, nrows=SAMPLE_ROWS, dtype=DTYPES, low_memory=False)
print(f"sample shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

In [ ]:
# Exact row count of the full file, without holding it in memory
total_rows = sum(len(chunk) for chunk in pd.read_csv(
    DATA_FILE, usecols=["Rndrng_NPI"], dtype=DTYPES, chunksize=250_000
))
print(f"full file: {total_rows:,} rows")

## 2. Schema profile

What columns exist, what type Pandas infers, and how populated each one is.

In [ ]:
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=True),
    "example": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
})
schema.sort_values("null_pct", ascending=False)

## 3. Null audit

The contract can only assert `not null` on columns that are *actually* complete.
Anything above 0% here has to stay out of the completeness checks.

In [ ]:
nulls = (df.isna().mean() * 100).round(3).sort_values(ascending=False)
complete = nulls[nulls == 0].index.tolist()
incomplete = nulls[nulls > 0]

print(f"{len(complete)} fully populated columns, {len(incomplete)} with nulls\n")
print("columns with nulls (%):")
print(incomplete.to_string())

In [ ]:
CANDIDATE_NOT_NULL = [
    "Rndrng_NPI",
    "Rndrng_Prvdr_Type",
    "Rndrng_Prvdr_State_Abrvtn",
    "Tot_Srvcs",
    "Tot_Mdcr_Pymt_Amt",
]

for col in CANDIDATE_NOT_NULL:
    print(f"{col:<30} nulls: {df[col].isna().sum():>6}")

## 4. `Rndrng_NPI` — identity column

The National Provider Identifier is the natural key. Two things to confirm
before the contract can rely on it: fixed 10-digit width, and one row per NPI.

In [ ]:
lengths = df["Rndrng_NPI"].str.len().value_counts()
print("NPI string lengths:")
print(lengths.to_string())

leading_zero = df["Rndrng_NPI"].str.startswith("0").sum()
print(f"\nNPIs with a leading zero: {leading_zero}")

In [ ]:
# Uniqueness across the whole file, not just the sample
npis = pd.concat([
    chunk["Rndrng_NPI"] for chunk in pd.read_csv(
        DATA_FILE, usecols=["Rndrng_NPI"], dtype=DTYPES, chunksize=250_000
    )
])
print(f"rows: {len(npis):,}   distinct NPIs: {npis.nunique():,}")
print(f"duplicated: {len(npis) - npis.nunique():,}")

## 5. `Rndrng_Prvdr_State_Abrvtn` — categorical domain

CMS uses more than the 50 states: territories, military APO/FPO codes, the
Freely Associated States, and `ZZ` as a catch-all for foreign addresses.
Enumerating what actually appears is how the accepted value set was built.

In [ ]:
states = df["Rndrng_Prvdr_State_Abrvtn"].value_counts(dropna=False)
print(f"{len(states)} distinct codes in sample\n")
print(states.to_string())

In [ ]:
FIFTY_STATES = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY",
}

non_state = states[~states.index.isin(FIFTY_STATES)]
print("codes outside the 50 states:")
print(non_state.to_string())
print(f"\nshare of rows: {non_state.sum() / len(df) * 100:.3f}%")

That long tail is why the state expectation carries `mostly=0.999` — the listed
set covers essentially everything, with 0.1% of headroom so a newly introduced
CMS code fails the build as a warning-sized deviation rather than a surprise.

## 6. Numeric ranges

Charges, allowed amounts, and payments should never be negative; service and
beneficiary counts should never be zero. Confirming that before asserting it.

In [ ]:
NUMERIC = [
    "Tot_Benes",
    "Tot_Srvcs",
    "Tot_Sbmtd_Chrg",
    "Tot_Mdcr_Alowd_Amt",
    "Tot_Mdcr_Pymt_Amt",
    "Tot_Mdcr_Stdzd_Amt",
]

df[NUMERIC].describe().T[["count", "min", "mean", "50%", "max"]]

In [ ]:
for col in NUMERIC:
    negatives = (df[col] < 0).sum()
    zeros = (df[col] == 0).sum()
    print(f"{col:<22} negative: {negatives:>6}   zero: {zeros:>6}")

## 7. Provider mix

Not asserted by the contract — cardinality here is too volatile year to year to
pin down — but it is the context for why `Rndrng_Prvdr_Type` is checked for
completeness rather than membership in a fixed set.

In [ ]:
provider_types = df["Rndrng_Prvdr_Type"].value_counts()
print(f"{provider_types.size} distinct provider types\n")
provider_types.head(15)

## 8. Findings → contract

Every expectation in `pipelines/contract.py` traces back to something above.
The contract has since grown well past what this notebook first motivated —
115 rules over 81 columns — so this is the derivation, not the full list.

| Observed here | Expectation |
|---|---|
| 12 columns are load-bearing for downstream analysis | `ExpectColumnToExist` x 12 |
| The file carries 81 columns, not 12 | `ExpectTableColumnsToMatchSet` locks the whole set |
| NPI, provider type, state, services, payment have 0% nulls | `ExpectColumnValuesToNotBeNull` x 5 |
| Demographic columns are partly null by design (suppression) | `ExpectColumnProportionOfNonNullValuesToBeBetween` |
| NPI is a fixed 10-character identifier | `ExpectColumnValueLengthsToEqual(10)`, plus `^\d{10}$` |
| ...and carries a Luhn check digit over the 80840 prefix | `ExpectColumnValuesToBeValidNpi` (custom) |
| No negative charges, allowed amounts, or payments | `ExpectColumnValuesToBeBetween(min=0)` |
| Every row has at least one service and one beneficiary | `ExpectColumnValuesToBeBetween(min=1)` |
| States fall in a known CMS code set, with a small foreign tail | `ExpectColumnValuesToBeInSet(mostly=0.999)` |
| Chronic condition percentages are top-coded at 75 | `ExpectColumnValuesToBeBetween(0, 75)` x 25 |
| Suppression markers are only `*` or `#` | `ExpectColumnValuesToBeInSet` |
| ~1.26M rows, stable year over year | `ExpectTableRowCountToBeBetween(1M, 15M)` |
| One row per NPI | `ExpectColumnValuesToBeUnique` |

### What single-column profiling cannot see

Everything above looks at one column at a time, and that is the blind spot.
A row can hold a Medicare payment ten times its own submitted charge and
satisfy every rule in the table.

CMS defines the allowed amount as the Medicare payment plus the deductible,
coinsurance and any third-party liability. Payment is therefore a *component*
of allowed and can never exceed it — an arithmetic invariant that only a pair
rule can express:

    Tot_Sbmtd_Chrg  >=  Tot_Mdcr_Alowd_Amt  >=  Tot_Mdcr_Pymt_Amt

Worth checking against the sample loaded above:

```python
(df["Tot_Mdcr_Alowd_Amt"] < df["Tot_Mdcr_Pymt_Amt"]).sum()   # identity: 0
(df["Tot_Sbmtd_Chrg"] < df["Tot_Mdcr_Alowd_Amt"]).sum()      # convention: a handful
```

The two halves behave differently, which is why the contract treats them
differently. The first held on all 1.26M rows across all three backends. The
second failed on 48 — Medicare pays the lesser of submitted and allowed, so a
provider billing under the fee schedule inverts it — so that half alone
carries `mostly=0.999`.

**Deliberately excluded:** `ExpectColumnValuesToBeOfType`. `int64`/`float64`
are Pandas dtype names, and `Rndrng_NPI` is `TEXT` in Postgres, so a type
assertion could not pass on all three backends — and running one suite
against all three is the point of the project.